# Part 1: CT scans to Pillar-0 embedding vectors

**Purpose**

Converts chest CT volumes into fixed-length numerical summaries (embeddings)
using the frozen Pillar-0 foundation model, and writes them to a single CSV.

This is stage one of a two-stage workflow:

| Stage | Notebook | Hardware | Produces |
|---|---|---|---|
| 1 | `Part1_Pillar0_Embeddings_to_CSV.ipynb` | GPU | `embeddings_*.csv` |
| 2 | `Part2_Train_and_Compare_Task_Heads.ipynb` | CPU (laptop) | metrics, plots, comparisons |

**This notebook supports two data sources. Pick one in Section 1.**

| | Path A | Path B |
|---|---|---|
| Dataset | CT-RATE | RSNA STR Pulmonary Embolism Detection |
| Contrast | Non-contrast chest CT | CTPA (contrast) |
| Labels | 18 report-derived findings | Study-level PE labels including `central_pe` |
| Run on | Google Colab | Kaggle Notebooks |
| Data access | Streamed per scan from HuggingFace | Pre-mounted, zero download |
| Use it for | Any of the 18 findings, quick pipeline checks | Central PE, replication of a published cohort |
| Cannot do | Pulmonary embolism (no PE label exists) | Findings other than PE |

Both paths converge on the same output format, so Part 2 reads either one
without modification.

**Path B must be run on Kaggle.** The RSNA DICOM archive is about 900 GB. It is
pre-mounted inside Kaggle notebooks, so there is nothing to download. Attempting
this on Colab or a laptop is not practical.


## 0. Check the GPU

Pillar-0 needs an NVIDIA GPU. One input volume is 11 x 256 x 256 x 256, roughly
3 GB before the model runs, so CPU execution is not practical.

- **Colab:** Runtime > Change runtime type > T4 GPU
- **Kaggle:** right sidebar > Accelerator > GPU T4 x2, and Internet > On

If this prints nothing, change the runtime before continuing.


In [ ]:
# IN:  nothing
# OUT: prints the GPU name and memory

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Choose the path

Set `PATH` to `"A"` or `"B"`. Every cell below branches on this one value.


In [ ]:
# IN:  nothing
# OUT: PATH, and the settings for whichever path was chosen

PATH = "B"        # "A" = CT-RATE on Colab, "B" = RSNA on Kaggle

# ---- Path A settings (CT-RATE) -------------------------------------------
A_REPO_ID = "ibrahimhamamci/CT-RATE"
A_SPLIT   = "train_fixed"     # v2 volumes, already intensity-corrected
A_LABEL   = "Cardiomegaly"    # any column in the multi-abnormality label file
A_N_SCANS = 400               # balanced, half positive and half negative
A_OUT     = "/content/drive/MyDrive/P3_embeddings/embeddings_pillar0_ctrate.csv"

# ---- Path B settings (RSNA) ----------------------------------------------
# The encode list is a CSV with columns: file_name, labels, subsets, study, series
# See README section "Building an encode list for Path B" to produce one.
B_ENCODE_LIST = None          # None = auto-discover in /kaggle/input/datasets
B_OUT         = "/kaggle/working/embeddings_pillar0_rsna.csv"

# ---- Fixed by the model, do not change -----------------------------------
MODEL_ID = "YalaLab/Pillar0-ChestCT"
DEPTH    = 256                # slices
SIZE     = 256                # in-plane
CHUNK    = 25                 # studies per preprocessing batch
SEED     = 42

OUT_CSV = A_OUT if PATH == "A" else B_OUT
print("path:", PATH, "| output:", OUT_CSV)

**Why the geometry is fixed.** `DEPTH` and `SIZE` come from Pillar-0's own
configuration (`pillar-finetune/configs/csv_dataset.yaml`: `num_images: 256`,
`img_size: [256, 256]`). The learned position embeddings reject other
geometries, and the internal merge ratio of 4 means the depth must be divisible
by 4 after patch embedding. A volume of 169 slices produces 22 patches, which is
not divisible by 4, and the model raises an einops error. Section 5 pads short
volumes to 256 to prevent this.


## 2. Install

Order matters. RAVE installs its own copy of `transformers`, so the pinned
version goes last to overwrite it. Newer versions fail to load this checkpoint
through `trust_remote_code`.

**On Colab, restart the session after this cell** (Runtime > Restart session),
then skip it and continue from Section 3. On Kaggle inside a committed run the
kernel is fresh anyway, so no restart is needed.


In [ ]:
# IN:  nothing
# OUT: installed packages

RAVE_DIR = "/kaggle/working/rave" if PATH == "B" else "/content/rave"

!git clone -q https://github.com/YalaLab/rave.git {RAVE_DIR}
!pip install -q {RAVE_DIR}
!pip install -q nibabel pydicom pylibjpeg pylibjpeg-libjpeg python-gdcm SimpleITK lz4 pandas numpy tqdm huggingface_hub
!pip install -q transformers==4.55.2

import transformers
print("transformers:", transformers.__version__, "| want 4.55.2")

## 3. Authentication

**Path A** needs a HuggingFace token for CT-RATE, which is free but gated.

1. Create an account at `huggingface.co/join`.
2. Visit `huggingface.co/datasets/ibrahimhamamci/CT-RATE` and accept the terms.
3. Create a **read** token at `huggingface.co/settings/tokens`.

**Path B** needs a HuggingFace token for Pillar-0 itself, which is also gated.

1. Visit `huggingface.co/YalaLab/Pillar0-ChestCT` and request access.
2. Create a **read** token at `huggingface.co/settings/tokens`.
3. On Kaggle: Add-ons > Secrets > Add secret, label `HF_TOKEN`, and confirm it
   is attached to this notebook. Never paste a token into a cell.

Path B also requires accepting the competition rules at
`kaggle.com/competitions/rsna-str-pulmonary-embolism-detection` before the data
will mount.


In [ ]:
# IN:  a HuggingFace read token
# OUT: authenticated session

from huggingface_hub import login

if PATH == "B":
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
else:
    login()      # paste the token when prompted

print("authenticated")

## 4. Preprocessing configuration

RAVE ships the official chest CT preprocessing recipe. We copy it and change two
things.

**Absolute exporter path.** RAVE resolves its exporter config relative to the
package directory, which breaks when installed via pip because it looks inside
`site-packages/configs/`. An absolute path bypasses the lookup.

**Lossless export.** The default is HEVC video compression, which needs ffmpeg
with the right codec and is lossy. `lz4_fast` is lossless, so Hounsfield values
survive exactly. That matters when the finding is a small filling defect.

The recipe itself is left untouched: resample to 1.25 mm isotropic, crop or pad
to 256 x 256 in-plane, take the middle 256 slices. Resampling to a fixed
physical scale is the important part. Naive slice selection with `linspace`
stretches the z axis by a different factor for every scan, which is acceptable
for a gross finding like cardiomegaly and not acceptable for a clot.


In [ ]:
# IN:  the cloned rave repo
# OUT: ct_chest_p3.yaml with absolute exporter path and lossless export

import yaml, os

work_dir = "/kaggle/working" if PATH == "B" else "/content"
CONFIG   = f"{work_dir}/ct_chest_p3.yaml"

cfg = yaml.safe_load(open(f"{RAVE_DIR}/configs/ct_chest.yaml"))
cfg["exporter_config"] = f"{RAVE_DIR}/configs/exporters/lz4_fast.yaml"
with open(CONFIG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(open(CONFIG).read())

## 5. Helper functions

`dicom_to_nifti` is only used by Path B. RSNA ships DICOM, and its
de-identification replaced `SeriesInstanceUID` with a 12-character hash rather
than a dotted numeric UID. GDCM validates UIDs and discards the file, so
SimpleITK's directory scanner reports zero series even though the images are
fine. pydicom does not validate, so reading the files directly and writing a
NIfTI sidesteps the problem entirely. It also gives explicit control over the
two things that must not be wrong: slice ordering, taken from
`ImagePositionPatient` rather than `InstanceNumber`, and the Hounsfield
conversion from `RescaleSlope` and `RescaleIntercept`.

`to_tensor` centre-pads short volumes with air (-1024 HU) and centre-crops long
ones, then applies all 11 CT windows. Both paths use it.


In [ ]:
# IN:  nothing
# OUT: dicom_to_nifti (Path B only), to_tensor (both paths)

import pydicom, numpy as np, SimpleITK as sitk, os, glob, warnings
import torch, rve
import torch.nn.functional as F
warnings.filterwarnings("ignore", category=UserWarning)   # RSNA UIDs are non-conformant

def dicom_to_nifti(src, dst):
    """One directory of DICOM slices to one HU NIfTI with correct voxel spacing."""
    ds = [pydicom.dcmread(f) for f in glob.glob(os.path.join(src, "*.dcm"))]

    def zpos(d):
        try:    return float(d.ImagePositionPatient[2])
        except: return float(d.InstanceNumber)
    ds.sort(key=zpos)                                   # geometry, not InstanceNumber

    slope = float(getattr(ds[0], "RescaleSlope", 1))
    inter = float(getattr(ds[0], "RescaleIntercept", -1024))
    vol = np.stack([d.pixel_array for d in ds]).astype(np.float32) * slope + inter

    py, px = [float(v) for v in ds[0].PixelSpacing]     # PixelSpacing is [row, col] = [y, x]
    z  = [zpos(d) for d in ds]
    sz = abs(float(np.median(np.diff(z)))) if len(z) > 1 else float(ds[0].SliceThickness)

    img = sitk.GetImageFromArray(vol.astype(np.int16))  # array order is (z, y, x)
    img.SetSpacing((px, py, sz or 1.0))                 # SetSpacing wants (x, y, z)
    sitk.WriteImage(img, dst)

def to_tensor(proc_path, depth=DEPTH):
    """Processed volume to model input (1, 11, depth, 256, 256). Returns original depth too."""
    vol = rve.load_sample(proc_path, device="cuda").float()      # (D, H, W) in HU
    d = vol.shape[0]
    if d < depth:                                                 # pad with air, centred
        lo = (depth - d) // 2
        vol = F.pad(vol, (0, 0, 0, 0, lo, depth - d - lo), value=-1024.0)
    elif d > depth:                                               # centre crop
        lo = (d - depth) // 2
        vol = vol[lo:lo + depth]
    return rve.apply_windowing(vol, "all", "CT").unsqueeze(0).float(), d

print("helpers ready")

## 6. Build the work list

**Path A** downloads only the label spreadsheet, which is small, then selects a
balanced set of scans. CT-RATE names volumes `split_patient_scan_reconstruction`,
so `train_1_a_1` is reconstruction 1 of scan `a` from patient 1, and the storage
path is derived from that name. One volume per patient is kept so no patient
appears twice.

**Path B** reads a prepared encode list naming which studies to process, and
resolves each to its DICOM directory under the mounted competition data.


In [ ]:
# IN:  the settings from Section 1
# OUT: work, a dataframe with one row per volume to encode

import pandas as pd, glob, os

if PATH == "A":
    from huggingface_hub import hf_hub_download

    lab = pd.read_csv(hf_hub_download(
        A_REPO_ID, f"dataset/multi_abnormality_labels/{A_SPLIT.split('_')[0]}_predicted_labels.csv",
        repo_type="dataset"))
    lab["patient"] = lab.VolumeName.str.split("_").str[:2].str.join("_")
    lab = lab.drop_duplicates("patient")               # one volume per patient

    pos = lab[lab[A_LABEL] == 1].sample(A_N_SCANS // 2, random_state=SEED)
    neg = lab[lab[A_LABEL] == 0].sample(A_N_SCANS // 2, random_state=SEED)
    work = pd.concat([pos, neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    work["file_name"] = work.VolumeName.str.replace(".nii.gz", "", regex=False)
    work["labels"]    = work[A_LABEL].astype(int)
    work["subsets"]   = ""                             # no predefined splits
    p = work.file_name.str.split("_")
    work["hf_path"] = ("dataset/" + A_SPLIT + "/" + p.str[0] + "_" + p.str[1] + "/"
                       + p.str[0] + "_" + p.str[1] + "_" + p.str[2] + "/"
                       + work.file_name + ".nii.gz")

else:
    ROOT = glob.glob("/kaggle/input/competitions/*")[0]
    path = B_ENCODE_LIST or glob.glob("/kaggle/input/datasets/**/*encode_list.csv", recursive=True)[0]
    work = pd.read_csv(path)
    work["dicom_dir"] = ROOT + "/train/" + work.study + "/" + work.series
    assert os.path.isdir(work.dicom_dir.iloc[0]), "DICOM path does not resolve"

print(PATH, "|", len(work), "volumes |", work.labels.value_counts().to_dict())
work.head(3)

## 7. Load the model

The model is frozen. `requires_grad` is switched off on every parameter so no
gradients are computed and no weights change. The encoder is a fixed translator
from images to numbers, and all learning happens in the classifier trained in
Part 2.

The output is 1152 numbers per scan regardless of what is being predicted. One
embedding run therefore supports any number of different findings, which is the
practical basis for the claim that additional task-specific models cost very
little once the first is built.


In [ ]:
# IN:  authenticated HuggingFace session
# OUT: the frozen Pillar-0 encoder on the GPU

from transformers import AutoModel

model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).cuda().eval()
for p in model.parameters():
    p.requires_grad = False

n = sum(p.numel() for p in model.parameters())
print(f"loaded {MODEL_ID} | {n/1e6:.1f}M parameters | frozen")

## 8. Smoke test

One volume end to end before committing hours of GPU time. The assertions are
the point: a silent geometry mismatch across hundreds of scans either crashes
partway through or, worse, does not crash.


In [ ]:
# IN:  the first row of work
# OUT: proof the whole chain produces a (1152,) vector

import subprocess, shutil, time
from huggingface_hub import hf_hub_download

smoke = f"{work_dir}/smoke"
shutil.rmtree(smoke, ignore_errors=True)
os.makedirs(f"{smoke}/nii", exist_ok=True)

r = work.iloc[0]
t0 = time.time()
if PATH == "A":
    src = hf_hub_download(A_REPO_ID, r.hf_path, repo_type="dataset", local_dir=f"{smoke}/nii")
else:
    src = f"{smoke}/nii/t.nii.gz"
    dicom_to_nifti(r.dicom_dir, src)
t_prep = time.time() - t0

pd.DataFrame({"series_path": [src]}).to_csv(f"{work_dir}/smoke.csv", index=False)
t0 = time.time()
subprocess.run(["vision-engine", "process", "--config", CONFIG,
                "--input-series-csv", f"{work_dir}/smoke.csv",
                "--output", f"{smoke}/proc", "--workers", "2"], capture_output=True)
t_rave = time.time() - t0

m = pd.read_csv(f"{smoke}/proc/mapping.csv")
x, depth = to_tensor(m.output_path.iloc[0])
assert tuple(x.shape) == (1, 11, DEPTH, SIZE, SIZE), f"wrong geometry: {x.shape}"

t0 = time.time()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    v = model.extract_vision_feats({"chest_ct": x}).float().flatten().cpu().numpy()
t_enc = time.time() - t0
assert v.shape == (1152,), v.shape

del x; torch.cuda.empty_cache()
shutil.rmtree(smoke, ignore_errors=True)

per = t_prep + t_rave + t_enc
print(f"passed | original depth {depth} | embedding {v.shape}")
print(f"fetch/convert {t_prep:.0f}s + rave {t_rave:.0f}s + encode {t_enc:.0f}s = {per:.0f}s per scan")
print(f"{len(work)} scans -> {per*len(work)/3600:.1f} hours")

## 9. The run

Volumes are processed in chunks. Each chunk is fetched or converted, handed to
RAVE once so preprocessing runs in parallel across CPU cores, embedded, then
deleted. Peak disk stays around a couple of gigabytes regardless of how many
scans are in the list.

**Resumable.** The output CSV is rewritten after every chunk and re-read at
startup, so an interrupted run continues where it stopped. This matters because
free GPU sessions disconnect.

**Input to output pairing comes from RAVE's `mapping.csv`**, not from matching
filenames. RAVE does not always derive its output name from the input filename,
so substring matching silently loses studies.

**On Kaggle, launch with Save Version > Save and Run All (Commit)** rather than
running interactively. A commit runs headless for up to 12 hours and persists
`/kaggle/working`, so the laptop can be closed. To resume a killed run, attach
the previous version's output as an input dataset and point the resume read at
it.


In [ ]:
# IN:  work + model
# OUT: OUT_CSV, resumable, with one row per successfully encoded volume

import subprocess, shutil, time
from huggingface_hub import hf_hub_download

NII, PROC = f"{work_dir}/tmp_nii", f"{work_dir}/tmp_proc"

rows, done = [], set()
if os.path.exists(OUT_CSV):
    old = pd.read_csv(OUT_CSV)
    rows, done = old.to_dict("records"), set(old.file_name)
    print("resuming,", len(done), "already done")

todo = work[~work.file_name.isin(done)].reset_index(drop=True)
print(len(todo), "remaining")
failures, t_start = [], time.time()

for i in range(0, len(todo), CHUNK):
    batch = todo.iloc[i:i + CHUNK]
    for d in (NII, PROC):
        shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)

    # --- fetch or convert, remembering the exact path per volume ---
    src = {}
    for r in batch.itertuples():
        p = f"{NII}/{r.file_name}.nii.gz"
        try:
            if PATH == "A":
                got = hf_hub_download(A_REPO_ID, r.hf_path, repo_type="dataset", local_dir=NII)
                src[os.path.abspath(got)] = r
            else:
                dicom_to_nifti(r.dicom_dir, p)
                src[os.path.abspath(p)] = r
        except Exception as e:
            failures.append((r.file_name, "fetch", str(e)[:80]))

    # --- preprocess the whole chunk at once ---
    pd.DataFrame({"series_path": list(src)}).to_csv(f"{work_dir}/chunk.csv", index=False)
    subprocess.run(["vision-engine", "process", "--config", CONFIG,
                    "--input-series-csv", f"{work_dir}/chunk.csv",
                    "--output", PROC, "--workers", "4"], capture_output=True)

    # --- pair input to output through mapping.csv, never by filename matching ---
    mp = f"{PROC}/mapping.csv"
    m = (pd.read_csv(mp) if os.path.exists(mp) and os.path.getsize(mp) > 1
         else pd.DataFrame(columns=["source_path", "output_path"]))
    got = {os.path.abspath(s): o for s, o in zip(m.source_path, m.output_path)}

    # --- embed ---
    for p, r in src.items():
        if p not in got:
            failures.append((r.file_name, "rave", "not in mapping")); continue
        try:
            x, depth = to_tensor(got[p])
            assert tuple(x.shape) == (1, 11, DEPTH, SIZE, SIZE)
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                v = model.extract_vision_feats({"chest_ct": x}).float().flatten().cpu().numpy()
            rows.append({"labels": r.labels,
                         "series_dir": getattr(r, "dicom_dir", getattr(r, "hf_path", "")),
                         "series_id": r.file_name,
                         "embedding": str(v.tolist()),
                         "file_name": r.file_name,
                         "subsets": r.subsets,
                         "orig_depth": depth})
            del x; torch.cuda.empty_cache()
        except Exception as e:
            failures.append((r.file_name, "embed", str(e)[:80]))

    pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
    el, n = time.time() - t_start, len(rows) - len(done)
    print(f"{len(rows)}/{len(work)} | {el/max(n,1):.1f}s/scan | "
          f"eta {(len(work)-len(rows))*el/max(n,1)/3600:.1f}h | fails {len(failures)}")

for d in (NII, PROC):
    shutil.rmtree(d, ignore_errors=True)
print("\ndone:", len(rows), "| failures:", len(failures))
for f in failures[:15]:
    print("  ", f)

## 10. Check the output

Run this before moving to Part 2. The failure modes worth catching: rows of all
zeros mean the encoder silently returned nothing, a near-zero minimum
per-dimension standard deviation means a dead embedding dimension, and a row
count well below the work list means studies were lost.

`orig_depth` is recorded because scan coverage varies. If most volumes are
heavily padded that is a characteristic of the cohort worth reporting, and it is
also worth confirming it does not predict the label on its own.


In [ ]:
# IN:  OUT_CSV
# OUT: a verdict on completeness and format

import numpy as np

df = pd.read_csv(OUT_CSV)
print(df.shape, "| expected", (len(work), 7))
print("labels    :", df.labels.value_counts().to_dict())
print("duplicates:", int(df.file_name.duplicated().sum()))

X = np.array([np.fromstring(s[1:-1], sep=",") for s in df.embedding])
print("\nX:", X.shape, "| NaN", int(np.isnan(X).sum()), "| all-zero rows", int((X == 0).all(1).sum()))
sd = X.std(0)
print("per-dim std min/median/max:", round(sd.min(), 4), round(float(np.median(sd)), 4), round(sd.max(), 4))

print("\noriginal depth min/median/max:", df.orig_depth.min(),
      int(df.orig_depth.median()), df.orig_depth.max(),
      "| padded:", int((df.orig_depth < DEPTH).sum()))

if df.subsets.astype(str).str.len().max() > 0:
    for tag in ["D1", "D2", "D3", "D4", "TEST"]:
        m = df.subsets.astype(str).str.contains(tag)
        if m.sum():
            print(f"{tag:5s} {m.sum():4d} volumes, {int(df[m].labels.sum()):3d} positive")

## 11. Retrieve the file

**Colab:** the CSV is already on Drive at the path in `OUT_CSV`.

**Kaggle:** open the notebook's Output tab and download
`embeddings_pillar0_rsna.csv`. Then move it next to Part 2:

```bash
mv ~/Downloads/embeddings_pillar0_rsna.csv path/to/frozen_pillar-0/
```

Size is roughly 15 KB per scan, so 956 scans is about 26 MB.

Continue in `Part2_Train_and_Compare_Task_Heads.ipynb`.
